In [ ]:
import numpy as np
import pandas as pd


def _load_df(datasources, start_date, end_date):
    import dai

    if isinstance(datasources, str) and datasources:
        table = datasources
    elif isinstance(datasources, dict):
        table = datasources.get('bigalpha_2026_stock_bar1m') or datasources.get('stock_bar1m') or datasources.get('bar1m') or 'bigalpha_2026_stock_bar1m'
    else:
        table = 'bigalpha_2026_stock_bar1m'

    sql = f'''WITH base AS (
        SELECT
            DATE(date) AS date,
            date AS ts,
            instrument,
            (EXTRACT(HOUR FROM date) * 60 + EXTRACT(MINUTE FROM date)) AS m,
            CASE
                WHEN bid_price1 IS NOT NULL AND ask_price1 IS NOT NULL
                THEN (CAST(bid_price1 AS DOUBLE) + CAST(ask_price1 AS DOUBLE)) / 2.0
                ELSE NULL
            END AS mid,
            CAST(volume AS DOUBLE) AS v,
            CAST(amount AS DOUBLE) AS amt
        FROM {table}
        WHERE DATE(date) >= DATE('{start_date}')
          AND DATE(date) <= DATE('{end_date}')
    ),
    minute AS (
        SELECT
            date, ts, instrument, m, mid, v, amt,
            mid / NULLIF(LAG(mid) OVER (PARTITION BY instrument, date ORDER BY ts), 0) - 1.0 AS ret
        FROM base
        WHERE mid IS NOT NULL
    ),
    open_daily AS (
        SELECT
            date,
            instrument,
            -1.0 * SUM(CASE WHEN m >= 570 AND m < 600 THEN v ELSE 0 END) / (SUM(v) + 1e-12) AS open_volume_exhaustion
        FROM minute
        GROUP BY date, instrument
    )
    SELECT minute.*, open_daily.open_volume_exhaustion
    FROM minute
    LEFT JOIN open_daily USING (date, instrument)
    ORDER BY date, instrument, ts'''

    df = dai.query(sql, compression=True, filters={'date': [start_date, end_date]})
    df = df if isinstance(df, pd.DataFrame) else df.df()
    df['date'] = pd.to_datetime(df['date'])
    df['instrument'] = df['instrument'].astype(str)
    for col in ['m', 'mid', 'v', 'amt', 'ret', 'open_volume_exhaustion']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.replace([np.inf, -np.inf], np.nan)
    return df


def _daily_rank(series):
    s = pd.to_numeric(series, errors='coerce')
    if s.notna().sum() == 0:
        return pd.Series([0.5] * len(s), index=s.index)
    filled = s.fillna(s.median()).fillna(0.0)
    return filled.rank(pct=True, method='average')


def _finish(raw):
    df = raw.copy()
    df['factor'] = pd.to_numeric(df['factor'], errors='coerce').replace([np.inf, -np.inf], np.nan)
    df['factor'] = df.groupby('date')['factor'].transform(lambda s: s.fillna(s.median()).fillna(0.5))
    df['factor'] = df.groupby('date')['factor'].rank(pct=True, method='average')
    df = df[['date', 'instrument', 'factor']].dropna(subset=['factor']).sort_values(['date', 'instrument']).reset_index(drop=True)
    print('factor shape', df.shape)
    print('columns', list(df.columns))
    print('null count', int(df['factor'].isna().sum()))
    if len(df):
        print('factor min/max', float(df['factor'].min()), float(df['factor'].max()))
        print('coverage', float(df['factor'].notna().mean()))
    else:
        print('factor min/max', None, None)
        print('coverage', 0.0)
    return df

def _raw_factor(df):
    out = []
    for (date, instrument), g in df.sort_values('ts').groupby(['date', 'instrument'], sort=False):
        g = g.sort_values('ts')
        tail = g.tail(30).copy()
        if len(tail) < 30:
            out.append((date, instrument, np.nan))
            continue
        tail['ret_1m'] = pd.to_numeric(tail['ret'], errors='coerce').fillna(0.0)
        tail['volume'] = pd.to_numeric(tail['v'], errors='coerce').fillna(0.0)
        tail_ret = tail['mid'].iloc[-1] / tail['mid'].iloc[0] - 1.0 if tail['mid'].notna().all() and tail['mid'].iloc[0] not in (0, None) else np.nan
        tail_vol_sum = tail['volume'].sum()
        day_vol_sum = pd.to_numeric(g['v'], errors='coerce').fillna(0.0).sum()
        minute_count = max(1, len(g))
        daily_avg_minute_vol = day_vol_sum / minute_count
        tail_vol_intensity = tail_vol_sum / (daily_avg_minute_vol * 30.0 + 1e-12) if day_vol_sum > 0 else 0.0
        total_vol = tail_vol_sum
        if total_vol <= 0:
            pos_vol_ratio = 0.5
        else:
            pos_vol_ratio = tail.loc[tail['ret_1m'] > 0, 'volume'].sum() / (total_vol + 1e-12)
        base_confirm = tail_ret * pos_vol_ratio if pd.notna(tail_ret) else np.nan
        confirm_raw = base_confirm if tail_vol_intensity > 1.0 else 0.0
        out.append((date, instrument, confirm_raw))
    return pd.DataFrame(out, columns=['date', 'instrument', 'confirm_raw'])


def main(datasources, start_date, end_date):
    df = _load_df(datasources, start_date, end_date)
    raw = _raw_factor(df)
    raw['confirm_rank'] = raw.groupby('date')['confirm_raw'].transform(_daily_rank)
    open_rank = df[['date', 'instrument', 'open_volume_exhaustion']].drop_duplicates().copy()
    open_rank['open_rank'] = open_rank.groupby('date')['open_volume_exhaustion'].rank(pct=True, method='average')
    raw = raw.merge(open_rank[['date', 'instrument', 'open_rank']], on=['date', 'instrument'], how='left')
    factor_raw = pd.Series(0.5, index=raw.index, dtype=float)
    mask = raw['confirm_rank'] > 0.80
    factor_raw.loc[mask] = raw.loc[mask, 'open_rank'] * raw.loc[mask, 'confirm_rank']
    raw['factor'] = factor_raw
    return _finish(raw)
